# Consistency Models
## 一致性模型：按照原论文顺序的完整学习笔记

**论文**：Consistency Models  
**作者**：Yang Song, Prafulla Dhariwal, Mark Chen, Ilya Sutskever  
**会议**：ICML 2023  
**原文**：[arXiv:2303.01469](https://arxiv.org/abs/2303.01469)  
**完整论文及附录**：[ICML PDF](https://proceedings.mlr.press/v202/song23a/song23a.pdf)

**阅读方式**：沿用上一篇《Stochastic Interpolants》学习笔记的组织方式：先总结全文核心结论，再严格按照论文的章节顺序讲解。每一章先说明作者想解决的问题，再定义符号、推导公式，最后安排针对性问答。数学知识在正文第一次出现时作简要解释，完整推导放在文末附录。

**内容标记**：

- **【论文】**：原论文提出的定义、方法、定理或实验结论。
- **【推导】**：对论文公式补充的中间计算步骤。
- **【延伸】**：为了加深理解增加的分析，不属于原论文直接证明的结论。

为方便对照，正文公式按照章节编号，例如式 (3.1)；附录公式使用 A.1、A.2 等编号。

---

# 0. 全文核心结论：先建立全局认识

Consistency Models 的目标是解决扩散生成模型的一个核心问题：

**传统 Diffusion Model 需要多次迭代才能生成样本，能否直接学习从任意噪声状态到最终数据的映射，从而实现一步生成？**

论文的答案是：可以。

它不再要求神经网络只预测局部 Score 或速度，而是直接学习一条 Probability Flow ODE 轨迹的终点。

## 结论一：Consistency Model 学习的是 ODE 的终点映射

假设有一个 Probability Flow ODE：

$$
\frac{dX_t}{dt}=v(X_t,t)
$$

其中：

- $X_t\in\mathbb R^d$：时间 $t$ 的状态。
- $v(x,t)$：ODE 的速度场。
- $t\in[\epsilon,T]$：噪声时间。
- $\epsilon>0$：最低噪声时刻。
- $T$：最高噪声时刻。

传统扩散模型需要通过 ODE Solver 从 $X_T$ 逐步积分到 $X_\epsilon$。

Consistency Model 则直接学习：

$$
\boxed{
f_\theta(x,t)\approx\Phi_{t\rightarrow\epsilon}(x)
}
\tag{0.1}
$$

其中：

$$
\Phi_{t\rightarrow\epsilon}(x)
$$

表示从时间 $t$ 的状态 $x$ 出发，沿 ODE 积分到 $\epsilon$ 时得到的终点。

因此，推理时：

$$
X_T\sim p_T
$$

$$
\boxed{
\widehat X_\epsilon=f_\theta(X_T,T)
}
\tag{0.2}
$$

只需要一次网络推理。

这里的关键区别是：

$$
\boxed{
\begin{aligned}
\text{Diffusion / Flow Matching：}&\quad \text{学习局部动力学}\\
\text{Consistency Model：}&\quad \text{学习全局终点映射}
\end{aligned}
}
$$

---

## 结论二：同一条 ODE 轨迹上的所有状态具有相同的终点预测

假设：

$$
X_{t_1},X_{t_2}
$$

属于同一条 ODE 轨迹。

由于它们最终到达相同的 $X_\epsilon$，因此：

$$
\boxed{
f_\theta(X_{t_1},t_1)
=
f_\theta(X_{t_2},t_2)
}
\tag{0.3}
$$

这就是论文所说的 **Self-Consistency（一致性）**。

同时，由于终点就是自身：

$$
\boxed{
f_\theta(x,\epsilon)=x
}
\tag{0.4}
$$

这两个条件共同确定了理想的一致性函数。

沿 ODE 轨迹求导，可以得到：

$$
\boxed{
\frac{\partial f}{\partial t}
+
J_xf(x,t)v(x,t)=0
}
\tag{0.5}
$$

其中：

$$
J_xf(x,t)=\frac{\partial f(x,t)}{\partial x}
$$

是网络对输入的 Jacobian。

式 (0.5) 是一致性模型背后的输运方程。

它说明：**一致性函数虽然接受时间和状态作为输入，但沿正确的 ODE 轨迹，其输出不随时间变化。**

---

## 结论三：Consistency Distillation（CD）通过教师 ODE 的局部轨迹训练

假设已经有预训练的 Score Network：

$$
s_\phi(x,t)\approx\nabla_x\log p_t(x)
$$

那么可以利用它定义 PF ODE，并通过数值积分构造：

$$
X_{t_{n+1}}
\longrightarrow
\widehat X_{t_n}
$$

这里两个状态位于同一条**数值近似的教师 ODE 轨迹**上。

随后训练：

$$
\boxed{
\mathcal L_{\mathrm{CD}}
=
\mathbb E\left[
d\left(
f_\theta(X_{t_{n+1}},t_{n+1}),
f_{\theta^-}(\widehat X_{t_n},t_n)
\right)
\right]
}
\tag{0.6}
$$

其中：

- $\theta$：当前训练模型。
- $\theta^-$：EMA Target Network。
- $d(\cdot,\cdot)$：距离函数。

**CD 的重要特点是：教师只需要执行一个局部积分步，而不需要对每个训练样本完整积分到终点。**

---

## 结论四：Consistency Training（CT）不需要预训练教师

CT 是论文更有理论价值的部分。

它直接使用：

$$
X_t=X_0+tZ
$$

其中：

$$
X_0\sim p_{\mathrm{data}},\qquad Z\sim\mathcal N(0,I)
$$

构造相邻状态：

$$
X_{t_{n+1}}=X_0+t_{n+1}Z
$$

$$
X_{t_n}=X_0+t_nZ
$$

然后训练：

$$
\boxed{
\mathcal L_{\mathrm{CT}}
=
\mathbb E\left[
d\left(
f_\theta(X_0+t_{n+1}Z,t_{n+1}),
f_{\theta^-}(X_0+t_nZ,t_n)
\right)
\right]
}
\tag{0.7}
$$

这里存在一个非常重要的细节：

**CT 构造的两个状态通常不在同一条 PF ODE 轨迹上。**

但是：

$$
\boxed{
v(x,t)
=
\mathbb E[Z\mid X_t=x]
}
\tag{0.8}
$$

即随机加噪路径的条件速度，在给定当前状态后的条件期望，等于 PF ODE 的边缘速度。

因此，CT 的局部更新可以在条件期望和连续时间极限下与 CD 建立联系。

这是 CT 不需要教师模型的数学基础。

---

## 结论五：CT 与 CD 的损失渐近接近，但不意味着有限步训练完全等价

论文 Theorem 2 在特定正则性条件下证明：

$$
\boxed{
\mathcal L_{\mathrm{CD}}^N
=
\mathcal L_{\mathrm{CT}}^N
+
o(\Delta t)
}
\tag{0.9}
$$

其中：

$$
\Delta t=\max_n(t_{n+1}-t_n)
$$

表示最大时间间隔。

但需要区分：

$$
\text{损失函数渐近接近}
$$

与：

$$
\text{两个模型的有限步最优解完全相同}
$$

这不是同一个结论。

原论文的连续时间分析还进一步研究了使用 Stop Gradient 时的梯度关系。

---

## 结论六：Consistency Model 支持一步生成，也支持多步改进

一步生成：

$$
\widehat X_\epsilon=f_\theta(X_T,T)
$$

多步生成则采用：

$$
\boxed{
\text{预测终点}
\rightarrow
\text{重新加噪}
\rightarrow
\text{再次预测终点}
}
\tag{0.10}
$$

注意：

**多步 Consistency Sampling 不是传统 ODE Solver。**

它不是简单地使用 Euler 或 Heun 积分速度场，而是多次调用终点预测网络。

---

# 1. Introduction：论文为什么提出 Consistency Models？

## 1.1 扩散模型的主要问题

扩散模型的生成过程通常可以表示为：

$$
X_T\rightarrow X_{t_{N-1}}\rightarrow\cdots\rightarrow X_\epsilon
$$

每一步都需要调用神经网络。

例如，在基于 Score 的 PF ODE 中：

$$
\frac{dX_t}{dt}
=
-\frac12g^2(t)\nabla_x\log p_t(X_t)
\tag{1.1}
$$

其中：

- $g(t)$：前向 SDE 的瞬时扩散系数。
- $p_t(x)$：时间 $t$ 的边缘密度。

神经网络近似：

$$
s_\phi(x,t)\approx\nabla_x\log p_t(x)
$$

于是：

$$
\frac{dX_t}{dt}
\approx
-\frac12g^2(t)s_\phi(X_t,t)
$$

通过 Euler、Heun 等 Solver 进行积分。

如果需要几十次甚至上百次网络评估，推理成本就很高。

论文希望将这个过程压缩为：

$$
X_T\xrightarrow{f_\theta}X_\epsilon
$$

只需要一次网络调用。

---

## 1.2 为什么不直接训练 $X_T\rightarrow X_0$？

一个自然想法是：

$$
\mathcal L
=
\mathbb E\|f_\theta(X_T,T)-X_0\|^2
$$

但这个目标存在问题。

对于加噪过程：

$$
X_T=X_0+TZ
$$

给定 $X_T=x$，真实数据 $X_0$ 通常并不唯一。

如果使用 MSE，那么最优解为：

$$
\boxed{
f_\theta^*(x,T)
=
\mathbb E[X_0\mid X_T=x]
}
\tag{1.2}
$$

这是后验均值。

但后验均值不一定等于 PF ODE 的终点。

也就是说：

$$
\boxed{
\mathbb E[X_0\mid X_T=x]
\ne
\Phi_{T\rightarrow\epsilon}(x)
}
\tag{1.3}
$$

一般情况下成立。

原因是：

- 后验均值来自概率回归。
- ODE 终点来自确定性动力系统。

它们的定义不同。

Consistency Models 不直接回归原始数据，而是学习由 ODE 轨迹确定的终点映射。

---

## 1.3 论文的解决思路

作者认为，如果一个模型能够满足：

$$
f_\theta(X_t,t)=X_\epsilon
$$

那么它就不需要知道从 $t$ 到 $\epsilon$ 的所有中间状态。

训练的核心因此变成：

**如何让同一条 ODE 轨迹上的不同状态预测相同终点？**

论文给出了两种方法：

1. 利用预训练扩散模型构造局部 ODE 轨迹，即 CD。
2. 利用随机加噪路径与边缘速度的条件期望关系，即 CT。

这两种方法将在第 4 节详细讨论。

### 本节问答

**Q1：Consistency Model 是不是把扩散模型的几十步积分压缩成一个大步 Euler？**

不是。

Euler 一步更新为：

$$
X_\epsilon
\approx
X_T+(\epsilon-T)v(X_T,T)
$$

它只使用初始状态处的局部速度。

Consistency Model 则直接学习：

$$
f_\theta(X_T,T)\approx\Phi_{T\rightarrow\epsilon}(X_T)
$$

理论上可以表达整个 ODE 积分过程的非线性效果。

**Q2：Consistency Model 与普通的 $x_0$-prediction 有什么区别？**

两者都输出数据空间中的状态，但训练目标不同。

普通去噪 MSE 的最优预测是后验均值；Consistency Model 的目标是 PF ODE 的终点。

---

# 2. Background：Score-Based Diffusion Models

这一节对应原论文的数学背景。

## 2.1 前向 SDE

【论文】

考虑：

$$
\boxed{
dX_t=\mu(X_t,t)\,dt+\sigma(t)\,dW_t
}
\tag{2.1}
$$

其中：

- $\mu(x,t)$：漂移项。
- $\sigma(t)$：扩散系数。
- $W_t$：标准布朗运动。
- $p_t(x)$：状态的边缘概率密度。

前向 SDE 从数据分布出发：

$$
X_0\sim p_{\mathrm{data}}
$$

逐渐将分布转换为容易采样的噪声分布。

---

## 2.2 Probability Flow ODE

【论文】

对于式 (2.1)，其 PF ODE 为：

$$
\boxed{
\frac{dX_t}{dt}
=
\mu(X_t,t)
-
\frac12\sigma^2(t)\nabla_x\log p_t(X_t)
}
\tag{2.2}
$$

它与原 SDE 在相同初始分布下具有相同的单时刻边缘分布。

### 【推导】为什么 PF ODE 具有相同的边缘分布？

SDE 的 Fokker–Planck 方程为：

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(\mu p_t)
+
\frac12\sigma^2(t)\Delta p_t
\tag{2.3}
$$

其中：

$$
\Delta p_t=\nabla\cdot\nabla p_t
$$

又因为：

$$
\nabla p_t=p_t\nabla\log p_t
$$

所以：

$$
\Delta p_t
=
\nabla\cdot(p_t\nabla\log p_t)
$$

代入：

$$
\begin{aligned}
\frac{\partial p_t}{\partial t}
&=
-\nabla\cdot(\mu p_t)\\
&\quad+
\frac12\sigma^2(t)
\nabla\cdot(p_t\nabla\log p_t)
\end{aligned}
$$

合并散度项：

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot
\left[
p_t\left(
\mu-\frac12\sigma^2(t)\nabla\log p_t
\right)
\right]
\tag{2.4}
$$

定义：

$$
v(x,t)
=
\mu(x,t)-\frac12\sigma^2(t)\nabla\log p_t(x)
$$

得到：

$$
\boxed{
\frac{\partial p_t}{\partial t}
+
\nabla\cdot(p_tv)=0
}
\tag{2.5}
$$

这正是确定性 ODE：

$$
\frac{dX_t}{dt}=v(X_t,t)
$$

所对应的连续性方程。

因此，在适当正则性与唯一性条件下，SDE 和 PF ODE 可以产生相同的边缘密度演化。

这里需要特别注意：

$$
\boxed{
p_t^{\mathrm{SDE}}=p_t^{\mathrm{ODE}}
}
$$

不代表：

$$
X_t^{\mathrm{SDE}}=X_t^{\mathrm{ODE}}
$$

两者的样本路径通常不同。

---

## 2.3 EDM 的 VE-SDE

【论文】

原论文使用 EDM 的噪声参数化：

$$
\mu(x,t)=0
$$

$$
\sigma(t)=\sqrt{2t}
$$

所以：

$$
\boxed{
dX_t=\sqrt{2t}\,dW_t
}
\tag{2.6}
$$

### 【推导】前向条件分布

对式 (2.6) 积分：

$$
X_t=X_0+\int_0^t\sqrt{2s}\,dW_s
$$

随机积分服从高斯分布，其协方差为：

$$
\int_0^t2s\,ds\,I=t^2I
$$

因此：

$$
\boxed{
X_t\mid X_0=x_0
\sim
\mathcal N(x_0,t^2I)
}
\tag{2.7}
$$

可以直接采样：

$$
\boxed{
X_t=X_0+tZ,\qquad Z\sim\mathcal N(0,I)
}
\tag{2.8}
$$

这里 $t$ 本身就是累计噪声标准差。

---

## 2.4 EDM 对应的 PF ODE

将：

$$
\mu=0,\qquad \sigma^2(t)=2t
$$

代入式 (2.2)：

$$
\frac{dX_t}{dt}
=
-\frac12(2t)\nabla\log p_t(X_t)
$$

得到：

$$
\boxed{
\frac{dX_t}{dt}
=
-t\nabla_x\log p_t(X_t)
}
\tag{2.9}
$$

定义：

$$
\boxed{
v^*(x,t)=-t\nabla_x\log p_t(x)
}
\tag{2.10}
$$

这就是后续 CD 和 CT 所讨论的目标速度场。

---

## 2.5 从去噪网络恢复 PF ODE 速度

EDM 的网络通常预测去噪结果：

$$
D_\phi(x,t)
$$

其理想目标为：

$$
D^*(x,t)=\mathbb E[X_0\mid X_t=x]
$$

由 Tweedie 公式：

$$
\nabla\log p_t(x)
=
\frac{\mathbb E[X_0\mid X_t=x]-x}{t^2}
$$

因此：

$$
\nabla\log p_t(x)
=
\frac{D^*(x,t)-x}{t^2}
$$

代入：

$$
v^*(x,t)=-t\nabla\log p_t(x)
$$

得到：

$$
\boxed{
v^*(x,t)
=
\frac{x-D^*(x,t)}{t}
}
\tag{2.11}
$$

因此，可以使用 EDM 去噪网络构造经验速度：

$$
\boxed{
v_\phi(x,t)=\frac{x-D_\phi(x,t)}{t}
}
\tag{2.12}
$$

这解释了为什么论文可以使用预训练 EDM 作为 CD 的教师。

### 本节问答

**Q1：为什么 EDM 的前向加噪可以一步完成，但反向生成仍然需要多步？**

因为前向 SDE 的条件分布：

$$
p_t(x_t\mid x_0)
$$

具有解析高斯形式。

但反向 PF ODE 的速度：

$$
v^*(x,t)=-t\nabla\log p_t(x)
$$

依赖复杂的边缘 Score，一般无法直接解析积分。

**Q2：既然去噪网络预测 $X_0$，为什么不能直接将它作为 Consistency Model？**

因为：

$$
D^*(x,t)=\mathbb E[X_0\mid X_t=x]
$$

而：

$$
f^*(x,t)=\Phi_{t\rightarrow\epsilon}(x)
$$

一般不相等。

去噪器学习后验均值；Consistency Model 学习确定性 ODE 终点。

**Q3：PF ODE 与 Flow Matching 的速度场有什么关系？**

对于相同的概率路径，如果使用：

$$
X_t=X_0+tZ
$$

构造 Conditional Flow Matching，则其边缘速度正好为：

$$
v^*(x,t)=\mathbb E[Z\mid X_t=x]
$$

这个关系将在第 4 节 CT 的推导中发挥关键作用。

---

# 3. Consistency Models：模型定义

这一节对应原论文第 3 节。

## 3.1 Consistency Function 的定义

【论文】

设：

$$
\frac{dX_t}{dt}=v(X_t,t)
$$

定义：

$$
\Phi_{t\rightarrow s}(x)
$$

为从时间 $t$ 的状态 $x$ 积分到时间 $s$ 的 ODE 流映射。

那么理想 Consistency Function 为：

$$
\boxed{
f^*(x,t)=\Phi_{t\rightarrow\epsilon}(x)
}
\tag{3.1}
$$

对于同一条轨迹上的任意两个状态：

$$
X_t,\quad X_s
$$

有：

$$
\boxed{
f^*(X_t,t)=f^*(X_s,s)
}
\tag{3.2}
$$

这就是 Self-Consistency Property。

---

## 3.2 边界条件

【论文】

当：

$$
t=\epsilon
$$

时，状态已经位于目标终点。

因此：

$$
\boxed{
f^*(x,\epsilon)=x
}
\tag{3.3}
$$

### 【推导】为什么边界条件必不可少？

如果只要求：

$$
f(X_t,t)=f(X_s,s)
$$

那么常数函数：

$$
f(x,t)=c
$$

也满足一致性。

但它显然不能实现生成任务。

加入：

$$
f(x,\epsilon)=x
$$

后，常数函数不再满足要求。

对于同一条 ODE 轨迹：

$$
X_\epsilon,X_{t_2},\ldots,X_T
$$

一致性条件给出：

$$
f(X_T,T)
=
f(X_{t_2},t_2)
=
f(X_\epsilon,\epsilon)
$$

而：

$$
f(X_\epsilon,\epsilon)=X_\epsilon
$$

所以：

$$
\boxed{
f(X_T,T)=X_\epsilon
}
\tag{3.4}
$$

这说明：

**边界条件负责确定输出是什么，一致性条件负责将终点信息传播到整条轨迹。**

---

## 3.3 Consistency Function 的 PDE

【推导】

因为：

$$
f^*(X_t,t)=X_\epsilon
$$

沿 ODE 轨迹不变，因此：

$$
\frac{d}{dt}f^*(X_t,t)=0
$$

使用链式法则：

$$
\frac{d}{dt}f^*(X_t,t)
=
\frac{\partial f^*}{\partial t}
+
J_xf^*\frac{dX_t}{dt}
$$

代入：

$$
\frac{dX_t}{dt}=v(X_t,t)
$$

得到：

$$
\boxed{
\partial_tf^*(x,t)
+
J_xf^*(x,t)v(x,t)=0
}
\tag{3.5}
$$

结合边界条件：

$$
\boxed{
\begin{cases}
\partial_tf^*+J_xf^*v=0\\
f^*(x,\epsilon)=x
\end{cases}
}
\tag{3.6}
$$

这就是一致性函数满足的输运方程。

### 直观理解

假设：

$$
f^*(x,t)
$$

预测的是未来最终轨迹。

当时间发生微小变化：

$$
t\rightarrow t+dt
$$

状态也会沿 ODE 变化：

$$
x\rightarrow x+v(x,t)dt
$$

因此：

$$
f^*(x+vdt,t+dt)
$$

应该与：

$$
f^*(x,t)
$$

相同。

Taylor 展开：

$$
f^*(x+vdt,t+dt)
=
f^*(x,t)
+
\partial_tf^*dt
+
J_xf^*vdt
+
o(dt)
$$

要求两者相等：

$$
\partial_tf^*dt+J_xf^*vdt=0
$$

除以 $dt$：

$$
\partial_tf^*+J_xf^*v=0
$$

因此，输运方程本质上就是自一致性条件的无穷小形式。

---

## 3.4 网络参数化

【论文】

原论文提出两种满足边界条件的方法。

第一种：

$$
f_\theta(x,t)=
\begin{cases}
x,&t=\epsilon\\
F_\theta(x,t),&t>\epsilon
\end{cases}
\tag{3.7}
$$

第二种：

$$
\boxed{
f_\theta(x,t)
=
c_{\mathrm{skip}}(t)x
+
c_{\mathrm{out}}(t)F_\theta(x,t)
}
\tag{3.8}
$$

要求：

$$
c_{\mathrm{skip}}(\epsilon)=1
$$

$$
c_{\mathrm{out}}(\epsilon)=0
$$

因此：

$$
f_\theta(x,\epsilon)=x
$$

论文采用第二种方式。

这种结构与 EDM 的网络预条件化形式具有相似性。

它能够通过结构设计严格满足边界条件，而不需要单独增加边界损失。

### 本节问答

**Q1：Consistency Model 学习的函数一定是可逆的吗？**

理想 ODE 流映射在满足存在唯一性条件时具有可逆性。

但是：

$$
f_\theta(x,t)
$$

是普通神经网络，论文没有显式施加可逆约束。

因此，实际网络不保证严格可逆。

**Q2：为什么只训练相邻时间的一致性，就能约束所有时间的终点？**

因为相邻时间关系可以沿时间网格传递：

$$
f(X_{t_N},t_N)
=
f(X_{t_{N-1}},t_{N-1})
=
\cdots
=
f(X_\epsilon,\epsilon)
$$

最后通过边界条件确定：

$$
f(X_\epsilon,\epsilon)=X_\epsilon
$$

但这是理想一致性关系；有限模型的误差会沿时间累积。

**Q3：这个 PDE 与 Flow Matching 的连续性方程是不是同一个？**

不是。

Flow Matching 的连续性方程：

$$
\partial_tp+\nabla\cdot(pv)=0
$$

描述概率密度如何随时间变化。

Consistency Function 的输运方程：

$$
\partial_tf+J_xfv=0
$$

描述一个函数值如何沿 ODE 轨迹保持不变。

两者使用同一个速度场，但作用对象不同。

---

# 4. Training Consistency Models：训练方法

这是原论文的核心章节。

## 4.1 Consistency Distillation（CD）

### 4.1.1 时间离散化

【论文】

将时间区间：

$$
[\epsilon,T]
$$

划分为：

$$
\epsilon=t_1<t_2<\cdots<t_N=T
$$

定义最大间隔：

$$
\Delta t
=
\max_{1\le n<N}(t_{n+1}-t_n)
\tag{4.1}
$$

对于每一对相邻时间：

$$
t_n<t_{n+1}
$$

需要构造同一条教师 ODE 轨迹上的两个状态。

---

### 4.1.2 教师 ODE 的局部积分

【论文】

首先采样：

$$
X_0\sim p_{\mathrm{data}}
$$

$$
Z\sim\mathcal N(0,I)
$$

构造：

$$
X_{t_{n+1}}=X_0+t_{n+1}Z
\tag{4.2}
$$

随后使用预训练教师的 ODE Solver：

$$
\boxed{
\widehat X_{t_n}
=
X_{t_{n+1}}
+
(t_n-t_{n+1})
\Phi(X_{t_{n+1}},t_{n+1};\phi)
}
\tag{4.3}
$$

其中：

$$
\Phi(\cdot;\phi)
$$

表示教师 ODE Solver 的单步更新方向。

如果使用 Euler：

$$
\Phi(x,t;\phi)=v_\phi(x,t)
$$

因此：

$$
\widehat X_{t_n}
=
X_{t_{n+1}}
+
(t_n-t_{n+1})v_\phi(X_{t_{n+1}},t_{n+1})
\tag{4.4}
$$

对于 EDM：

$$
v_\phi(x,t)=\frac{x-D_\phi(x,t)}{t}
$$

所以：

$$
\boxed{
\widehat X_{t_n}
=
X_{t_{n+1}}
+
\frac{t_n-t_{n+1}}{t_{n+1}}
\left[
X_{t_{n+1}}
-
D_\phi(X_{t_{n+1}},t_{n+1})
\right]
}
\tag{4.5}
$$

这是 Euler 教师的完整更新公式。

---

### 4.1.3 CD Loss

【论文】

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{CD}}^N
(\theta,\theta^-;\phi)
=
\mathbb E\Big[
\lambda(t_n)
d\big(
&f_\theta(X_{t_{n+1}},t_{n+1}),\\
&f_{\theta^-}(\widehat X_{t_n},t_n)
\big)
\Big]
\end{aligned}
}
\tag{4.6}
$$

其中：

- $\lambda(t_n)$：时间权重。
- $d$：距离函数。
- $\theta^-$：Target Network 参数。

论文采用 EMA：

$$
\boxed{
\theta^-
\leftarrow
\mu\theta^-+(1-\mu)\theta
}
\tag{4.7}
$$

Target Network 不接收梯度。

---

### 4.1.4 Theorem 1：CD 的误差阶

【论文】

设教师 ODE Solver 为 $p$ 阶方法。

在论文给定的正则性条件、零 CD Loss 和 Online/Target 参数一致等假设下：

$$
\boxed{
\sup_{n,x}
\left\|
f_\theta(x,t_n)-f(x,t_n;\phi)
\right\|
=
O((\Delta t)^p)
}
\tag{4.8}
$$

其中：

$$
f(x,t;\phi)
$$

表示教师经验 PF ODE 的精确终点映射。

### 【推导】为什么误差是 $O((\Delta t)^p)$？

设：

$$
h=t_{n+1}-t_n
$$

对于 $p$ 阶 ODE Solver，单步局部截断误差通常为：

$$
O(h^{p+1})
$$

如果时间区间长度固定，那么积分步数约为：

$$
N\sim O(1/h)
$$

累计误差为：

$$
O(1/h)\cdot O(h^{p+1})
=
O(h^p)
$$

因此：

$$
\boxed{
\text{全局误差}=O(h^p)
}
$$

对于 Euler：

$$
p=1
$$

得到：

$$
O(h)
$$

对于 Heun：

$$
p=2
$$

得到：

$$
O(h^2)
$$

这也是论文实验中使用 Heun 教师的重要原因。

### 本节问答

**Q1：CD 为什么不直接用教师完整生成的终点作为监督？**

可以这样设计，但完整积分成本较高。

CD 只需要局部 ODE 更新：

$$
X_{t_{n+1}}\rightarrow\widehat X_{t_n}
$$

通过相邻状态的一致性和边界条件，就可以学习终点映射。

**Q2：CD 的误差界是不是说明模型一定比教师好？**

不是。

它说明模型可以在理想条件下逼近教师经验 ODE 的终点映射。

如果教师本身存在 Score 误差，CD 不会自动消除这些误差。

---

## 4.2 Consistency Training（CT）

这一部分是论文最值得深入研究的地方。

### 4.2.1 不使用教师 ODE

【论文】

CT 直接采样：

$$
X_0\sim p_{\mathrm{data}}
$$

$$
Z\sim\mathcal N(0,I)
$$

构造：

$$
X_{t_{n+1}}=X_0+t_{n+1}Z
$$

$$
X_{t_n}=X_0+t_nZ
$$

损失：

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{CT}}^N
(\theta,\theta^-)
=
\mathbb E\Big[
\lambda(t_n)
d\big(
&f_\theta(X_0+t_{n+1}Z,t_{n+1}),\\
&f_{\theta^-}(X_0+t_nZ,t_n)
\big)
\Big]
\end{aligned}
}
\tag{4.9}
$$

注意：

这里不需要：

$$
s_\phi(x,t)
$$

也不需要 ODE Solver。

---

### 4.2.2 关键问题：两个状态并不在同一条 PF ODE 轨迹上

定义：

$$
h=t_{n+1}-t_n
$$

那么：

$$
X_{t_n}=X_{t_{n+1}}-hZ
\tag{4.10}
$$

而 PF ODE 的正确 Euler 更新是：

$$
\widehat X_{t_n}
=
X_{t_{n+1}}
-
hv^*(X_{t_{n+1}},t_{n+1})
\tag{4.11}
$$

通常：

$$
Z\ne v^*(X_t,t)
$$

所以：

$$
X_{t_n}\ne\widehat X_{t_n}
$$

CT 的随机直线路径与 PF ODE 轨迹不同。

---

### 4.2.3 【推导】为什么边缘速度等于条件期望？

从：

$$
X_t=X_0+tZ
$$

出发。

条件分布：

$$
p_t(x\mid x_0)
=
\frac1{(2\pi t^2)^{d/2}}
\exp\left(
-\frac{\|x-x_0\|^2}{2t^2}
\right)
\tag{4.12}
$$

取对数：

$$
\log p_t(x\mid x_0)
=
-\frac d2\log(2\pi t^2)
-
\frac{\|x-x_0\|^2}{2t^2}
$$

对 $x$ 求梯度：

$$
\nabla_x\log p_t(x\mid x_0)
=
-\frac{x-x_0}{t^2}
\tag{4.13}
$$

边缘密度：

$$
p_t(x)
=
\int p_t(x\mid x_0)p_{\mathrm{data}}(x_0)\,dx_0
$$

求梯度：

$$
\nabla p_t(x)
=
\int
\nabla p_t(x\mid x_0)
p_{\mathrm{data}}(x_0)\,dx_0
$$

利用：

$$
\nabla p_t(x\mid x_0)
=
p_t(x\mid x_0)
\nabla\log p_t(x\mid x_0)
$$

得到：

$$
\begin{aligned}
\nabla\log p_t(x)
&=
\frac{\nabla p_t(x)}{p_t(x)}\\
&=
\mathbb E\left[
-\frac{x-X_0}{t^2}
\middle|X_t=x
\right]
\end{aligned}
$$

因为：

$$
x-X_0=tZ
$$

所以：

$$
\nabla\log p_t(x)
=
-\frac1t\mathbb E[Z\mid X_t=x]
$$

两边乘以 $-t$：

$$
\boxed{
-t\nabla\log p_t(x)
=
\mathbb E[Z\mid X_t=x]
}
\tag{4.14}
$$

结合 PF ODE：

$$
v^*(x,t)=-t\nabla\log p_t(x)
$$

最终：

$$
\boxed{
v^*(x,t)=\mathbb E[Z\mid X_t=x]
}
\tag{4.15}
$$

这是 CT 的核心恒等式。

---

### 4.2.4 【推导】CT 的随机更新为什么能够近似 CD？

令：

$$
X=X_{t_{n+1}},\qquad t=t_{n+1}
$$

那么 CT 的低噪声状态：

$$
Y_{\mathrm{CT}}=X-hZ
$$

CD 的 Euler 状态：

$$
Y_{\mathrm{CD}}=X-hv^*(X,t)
$$

固定：

$$
X=x
$$

取条件期望：

$$
\begin{aligned}
\mathbb E[Y_{\mathrm{CT}}\mid X=x]
&=
\mathbb E[x-hZ\mid X=x]\\
&=
x-h\mathbb E[Z\mid X=x]\\
&=
x-hv^*(x,t)
\end{aligned}
$$

因此：

$$
\boxed{
\mathbb E[Y_{\mathrm{CT}}\mid X=x]
=
Y_{\mathrm{CD}}
}
\tag{4.16}
$$

但是，神经网络是非线性的：

$$
\mathbb E[f(Y_{\mathrm{CT}})]
\ne
f(\mathbb E[Y_{\mathrm{CT}}])
$$

因此，还需要 Taylor 展开。

---

### 4.2.5 【推导】Target Network 的 Taylor 展开

定义：

$$
g(x,t)=f_{\theta^-}(x,t)
$$

CT Target：

$$
g(x-hZ,t-h)
$$

在 $(x,t)$ 附近展开：

$$
\begin{aligned}
g(x-hZ,t-h)
={}&g(x,t)\\
&-hJ_xg(x,t)Z\\
&-h\partial_tg(x,t)\\
&+O(h^2)
\end{aligned}
\tag{4.17}
$$

对：

$$
X_t=x
$$

取条件期望：

$$
\begin{aligned}
&\mathbb E[g(x-hZ,t-h)\mid X_t=x]\\
&=
g(x,t)
-hJ_xg\,\mathbb E[Z\mid X_t=x]\\
&\quad-h\partial_tg
+O(h^2)
\end{aligned}
$$

代入：

$$
\mathbb E[Z\mid X_t=x]=v^*(x,t)
$$

得到：

$$
\boxed{
\begin{aligned}
&\mathbb E[g(x-hZ,t-h)\mid X_t=x]\\
&=
g(x,t)
-h\left[
\partial_tg+J_xg\,v^*
\right]
+O(h^2)
\end{aligned}
}
\tag{4.18}
$$

而 CD 的 Target：

$$
g(x-hv^*,t-h)
$$

同样展开：

$$
\boxed{
g(x-hv^*,t-h)
=
g(x,t)
-h\left[
\partial_tg+J_xg\,v^*
\right]
+O(h^2)
}
\tag{4.19}
$$

因此：

$$
\boxed{
\mathbb E[
g(Y_{\mathrm{CT}},t-h)
\mid X_t=x
]
=
g(Y_{\mathrm{CD}},t-h)+O(h^2)
}
\tag{4.20}
$$

这里的余项估计需要适当的光滑性与矩条件。

这说明：

**CT 与 CD 的 Target 在条件期望下具有相同的一阶变化。**

---

### 4.2.6 Theorem 2：CD Loss 与 CT Loss 的渐近关系

【论文】

在教师 Score 精确、使用 Euler Solver、距离函数与网络满足相应正则性条件等假设下：

$$
\boxed{
\mathcal L_{\mathrm{CD}}^N
=
\mathcal L_{\mathrm{CT}}^N
+
o(\Delta t)
}
\tag{4.21}
$$

这个结论可以通过对损失函数进行 Taylor 展开理解。

定义：

$$
A=f_\theta(x,t)
$$

考虑：

$$
d(A,g(x-hu,t-h))
$$

一阶展开：

$$
\begin{aligned}
d(A,g(x-hu,t-h))
={}&d(A,g(x,t))\\
&-h\,\partial_2d(A,g(x,t))
\left[
\partial_tg+J_xg\,u
\right]\\
&+O(h^2)
\end{aligned}
\tag{4.22}
$$

其中：

$$
\partial_2d
$$

表示距离函数对第二个输入的导数。

对于 CT：

$$
u=Z
$$

对于 CD：

$$
u=v^*
$$

因为：

$$
\mathbb E[Z\mid X_t=x]=v^*
$$

所以两个损失的一阶项相同。

因此，当时间间隔趋于零时，二者的差异是高阶小量。

但需要注意：

**这不意味着有限时间间隔下 CT 与 CD 完全相同，也不意味着两个优化问题必然具有相同的最优解。**

---

### 4.2.7 CT 的时间网格调度

【论文】

论文发现：

- 时间间隔大：训练信号较强，但近似偏差较大。
- 时间间隔小：近似更准确，但训练信号较弱。

因此，CT 采用逐渐增加时间离散点数量的策略。

定义训练迭代：

$$
k=0,\ldots,K
$$

论文使用：

$$
\boxed{
N(k)=
\left\lceil
\sqrt{
\frac{k}{K}
\left[
(s_1+1)^2-s_0^2
\right]
+s_0^2
}
-1
\right\rceil+1
}
\tag{4.23}
$$

其中：

- $s_0$：初始离散规模。
- $s_1$：最终离散规模。
- $K$：总训练步数。

同时调整 EMA：

$$
\boxed{
\mu(k)
=
\exp\left(
\frac{s_0\log\mu_0}{N(k)}
\right)
}
\tag{4.24}
$$

随着：

$$
N(k)\uparrow
$$

Target Network 更新逐渐变慢。

### 本节问答

**Q1：CT 为什么可以不用教师 Score Network？**

因为：

$$
v^*(x,t)=\mathbb E[Z\mid X_t=x]
$$

CT 使用随机速度 $Z$，其条件期望就是正确的 PF ODE 速度。

因此，CT 可以通过随机样本构造期望意义下正确的局部训练信号。

**Q2：CT 使用同一个 $Z$，是不是说明两个状态属于同一条 PF ODE 轨迹？**

不是。

共享噪声路径：

$$
X_t=X_0+tZ
$$

是一条随机条件路径。

PF ODE 使用的是：

$$
v^*(x,t)=\mathbb E[Z\mid X_t=x]
$$

两者通常不同。

**Q3：为什么 CT 的两个时间点必须使用同一个噪声？**

因为共享噪声时：

$$
X_{t_n}=X_{t_{n+1}}-hZ
$$

两个状态具有明确的局部增量关系。

如果使用独立噪声：

$$
Z_1,\quad Z_2
$$

则：

$$
X_{t_{n+1}}-X_{t_n}
=
t_{n+1}Z_1-t_nZ_2
$$

即使时间间隔趋近于零，两个状态也可能相距很远。

这就无法建立上述局部 Taylor 展开。

**Q4：Theorem 2 证明 CT 一定能够训练到正确的 ODE 终点吗？**

没有。

它证明的是训练目标之间的渐近关系。

实际训练是否收敛，还取决于模型容量、优化过程、EMA、时间调度等因素。

---

# 5. Sampling from Consistency Models

## 5.1 一步生成

【论文】

采样：

$$
X_T\sim\mathcal N(0,T^2I)
$$

然后：

$$
\boxed{
\widehat X_\epsilon=f_\theta(X_T,T)
}
\tag{5.1}
$$

只需要一次网络调用。

---

## 5.2 多步生成

【论文】

首先：

$$
\widehat X_\epsilon^{(1)}
=
f_\theta(X_T,T)
$$

然后选择中间噪声水平：

$$
\tau_1
$$

重新加噪：

$$
\boxed{
\widetilde X_{\tau_1}
=
\widehat X_\epsilon^{(1)}
+
\sqrt{\tau_1^2-\epsilon^2}Z_1
}
\tag{5.2}
$$

其中：

$$
Z_1\sim\mathcal N(0,I)
$$

再次预测终点：

$$
\boxed{
\widehat X_\epsilon^{(2)}
=
f_\theta(\widetilde X_{\tau_1},\tau_1)
}
\tag{5.3}
$$

重复这个过程。

### 【推导】为什么重新加噪的标准差是 $\sqrt{\tau^2-\epsilon^2}$？

VE-SDE：

$$
dX_t=\sqrt{2t}\,dW_t
$$

从：

$$
\epsilon\rightarrow\tau
$$

的新增噪声协方差：

$$
\int_\epsilon^\tau2t\,dt\,I
$$

计算：

$$
\int_\epsilon^\tau2t\,dt
=
\tau^2-\epsilon^2
$$

因此：

$$
X_\tau\mid X_\epsilon
\sim
\mathcal N(X_\epsilon,(\tau^2-\epsilon^2)I)
$$

所以：

$$
\boxed{
X_\tau
=
X_\epsilon
+
\sqrt{\tau^2-\epsilon^2}Z
}
\tag{5.4}
$$

这就是多步 Consistency Sampling 的加噪公式。

### 本节问答

**Q1：多步 Consistency Sampling 是不是 ODE 积分？**

不是。

ODE Solver 逐步更新状态：

$$
X_{t+h}=X_t+h\,v(X_t,t)
$$

Consistency Sampling 每一步都预测终点，然后重新加噪。

**Q2：既然模型可以一步生成，为什么多步生成效果通常更好？**

因为实际网络对终点映射存在近似误差。

重新加噪并再次预测，可以使模型在不同噪声水平上重新调整生成结果。

但多步改进属于经验现象，并不保证任意增加步数都会改善质量。

---

# 6. Applications：Zero-Shot Image Editing

原论文还研究了一致性模型的图像编辑能力。

## 6.1 核心思想

Consistency Model 可以从任意噪声水平预测终点：

$$
f_\theta(x_t,t)
$$

因此，可以在采样过程中加入观测约束。

例如图像修复：

设：

$$
y
$$

为观测图像。

定义：

$$
M_i=
\begin{cases}
1,&\text{未知像素}\\
0,&\text{已知像素}
\end{cases}
$$

先生成：

$$
\widehat x=f_\theta(x_t,t)
$$

然后施加约束：

$$
\boxed{
\widehat x_{\mathrm{constrained}}
=
(1-M)\odot y+M\odot\widehat x
}
\tag{6.1}
$$

接着重新加噪，再次调用 Consistency Model。

这样就可以逐步满足已知图像约束。

论文还展示了：

- 图像上色。
- 超分辨率。
- 图像修复。
- 笔画引导生成。
- 图像去噪。
- 潜在空间插值。

这些任务不需要分别训练专门的编辑模型。

### 本节问答

**Q1：Zero-Shot Editing 是不是说明模型天然知道所有图像编辑任务？**

不是。

它利用的是预训练生成先验，以及推理时加入的观测约束。

对于不符合训练分布的编辑任务，效果仍然可能下降。

**Q2：为什么需要多次重新加噪？**

因为一次终点预测未必同时满足生成分布约束和观测约束。

重新加噪并再次预测，可以让模型在两种约束之间反复调整。

---

# 7. Experiments：实验结果

## 7.1 实验任务

论文主要在以下数据集上实验：

- CIFAR-10。
- ImageNet $64\times64$。
- LSUN Bedroom $256\times256$。
- LSUN Cat $256\times256$。

主要评价指标是 FID。

FID 越低，说明生成分布与真实数据分布在所用特征空间中的差异越小。

---

## 7.2 CD 与 CT 的结果

论文报告的代表性结果如下：

| 数据集 | 方法 | 1 NFE FID | 2 NFE FID |
|---|---|---:|---:|
| CIFAR-10 | CD | 3.55 | 2.93 |
| CIFAR-10 | CT | 8.70 | 5.83 |
| ImageNet $64\times64$ | CD | 6.20 | 4.70 |
| ImageNet $64\times64$ | CT | 13.0 | 11.1 |

可以得到三个结论。

**第一，CD 的生成质量明显优于 CT。**

这说明准确的教师 ODE 轨迹监督能够显著改善一致性模型。

**第二，多步生成通常优于一步生成。**

这说明实际网络并不能完美学习全局终点映射。

**第三，CT 不需要预训练教师，也能实现一步生成。**

这是论文的重要贡献。

---

## 7.3 训练策略的消融

论文研究了：

- 不同距离函数。
- Euler 与 Heun。
- 时间离散点数量。
- EMA Target Network。
- 时间调度。
- 不同训练方式。

主要发现：

CD 更依赖教师模型与 ODE Solver 的质量。

CT 更依赖时间网格、EMA 和训练稳定性。

在图像任务中，LPIPS 距离往往比简单逐像素 L2 更有效。

### 本节问答

**Q1：为什么 CT 的 FID 明显比 CD 差？**

CT 使用的相邻状态并非真正的 PF ODE 轨迹状态。

它只能通过条件期望和小时间间隔近似正确的局部动力学。

因此，CT 存在有限步偏差，同时优化难度也更高。

**Q2：这些结果是否说明 CD 一定优于所有扩散采样器？**

不能。

论文证明了 CD 在少步生成方面的竞争力，但不同方法的模型规模、训练预算和 NFE 不同，不能脱离具体实验设置进行绝对比较。

---

# 8. Conclusion：论文总结

原论文提出了一个不同于传统扩散模型的生成建模方式。

传统扩散模型：

$$
\boxed{
\text{学习局部 Score 或速度}
\rightarrow
\text{多步积分}
\rightarrow
\text{生成样本}
}
$$

Consistency Model：

$$
\boxed{
\text{学习 ODE 终点映射}
\rightarrow
\text{一步或少步生成}
}
$$

论文的核心创新并不是简单减少采样步数，而是重新定义神经网络的学习对象。

从：

$$
v_\theta(x,t)
$$

变成：

$$
f_\theta(x,t)
$$

从学习局部动力学，变成学习全局流映射。

其中：

- CD 利用教师 ODE 的局部轨迹。
- CT 利用条件速度与边缘速度之间的期望关系。
- 边界条件确定终点。
- 自一致性约束使终点信息沿轨迹传播。

---

# 附录 A：数学基础补充

下面集中补充正文中使用的关键数学工具。

## A.1 条件期望为什么会得到最优 MSE 回归函数？

设：

$$
Y
$$

是监督目标，$X$ 是模型输入。

希望寻找函数：

$$
f(X)
$$

最小化：

$$
\mathcal L(f)
=
\mathbb E\|f(X)-Y\|^2
\tag{A.1}
$$

根据全期望公式：

$$
\mathcal L(f)
=
\mathbb E_X
\left[
\mathbb E[
\|f(X)-Y\|^2\mid X
]
\right]
$$

固定：

$$
X=x
$$

定义：

$$
m(x)=\mathbb E[Y\mid X=x]
$$

那么：

$$
f(x)-Y
=
[f(x)-m(x)]+[m(x)-Y]
$$

平方：

$$
\begin{aligned}
\|f(x)-Y\|^2
={}&\|f(x)-m(x)\|^2\\
&+\|m(x)-Y\|^2\\
&+2[f(x)-m(x)]^T[m(x)-Y]
\end{aligned}
$$

取条件期望。

由于：

$$
\mathbb E[m(x)-Y\mid X=x]=0
$$

交叉项消失。

因此：

$$
\begin{aligned}
&\mathbb E[\|f(x)-Y\|^2\mid X=x]\\
&=
\|f(x)-m(x)\|^2
+
\mathbb E[\|m(x)-Y\|^2\mid X=x]
\end{aligned}
$$

第二项与 $f$ 无关。

所以最优解为：

$$
\boxed{
f^*(x)=\mathbb E[Y\mid X=x]
}
\tag{A.2}
$$

**本文用途**：

解释为什么直接用 $X_0$ 做 MSE 监督得到的是后验均值，而不是 ODE 终点。

---

## A.2 Tweedie 公式

设：

$$
X=X_0+\sigma Z
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

那么：

$$
p(x\mid x_0)
=
\mathcal N(x;x_0,\sigma^2I)
$$

边缘密度：

$$
p(x)
=
\int p(x\mid x_0)p_{\mathrm{data}}(x_0)\,dx_0
$$

对 $x$ 求梯度：

$$
\nabla p(x)
=
\int
\nabla p(x\mid x_0)p_{\mathrm{data}}(x_0)\,dx_0
$$

而：

$$
\nabla p(x\mid x_0)
=
-\frac{x-x_0}{\sigma^2}p(x\mid x_0)
$$

因此：

$$
\nabla p(x)
=
-\frac1{\sigma^2}
\int
(x-x_0)p(x\mid x_0)p_{\mathrm{data}}(x_0)\,dx_0
$$

两边除以：

$$
p(x)
$$

得到：

$$
\nabla\log p(x)
=
-\frac1{\sigma^2}
\mathbb E[x-X_0\mid X=x]
$$

所以：

$$
\boxed{
\nabla\log p(x)
=
\frac{\mathbb E[X_0\mid X=x]-x}{\sigma^2}
}
\tag{A.3}
$$

整理：

$$
\boxed{
\mathbb E[X_0\mid X=x]
=
x+\sigma^2\nabla\log p(x)
}
\tag{A.4}
$$

这就是 Tweedie 公式。

**本文用途**：

从 EDM 的去噪预测恢复 Score 与 PF ODE 速度。

---

## A.3 多元 Taylor 展开

设：

$$
f:\mathbb R^d\times\mathbb R\rightarrow\mathbb R^m
$$

输入为：

$$
(x,t)
$$

考虑微小变化：

$$
(\Delta x,\Delta t)
$$

一阶 Taylor 展开：

$$
\boxed{
f(x+\Delta x,t+\Delta t)
=
f(x,t)
+
J_xf\,\Delta x
+
\partial_tf\,\Delta t
+
O(\|\Delta x\|^2+|\Delta t|^2)
}
\tag{A.5}
$$

对于 CT：

$$
\Delta x=-hZ
$$

$$
\Delta t=-h
$$

所以：

$$
\boxed{
f(x-hZ,t-h)
=
f(x,t)
-hJ_xfZ
-h\partial_tf
+O(h^2)
}
\tag{A.6}
$$

这就是 CT 理论推导中最常使用的公式。

需要注意：如果 $Z$ 是无界随机变量，余项在取期望后仍为 $O(h^2)$，还需要适当的导数有界性与随机变量矩条件。

---

## A.4 条件期望与非线性函数为什么不能交换？

一般情况下：

$$
\boxed{
\mathbb E[f(X)]\ne f(\mathbb E[X])
}
\tag{A.7}
$$

例如：

$$
f(x)=x^2
$$

那么：

$$
\mathbb E[f(X)]=\mathbb E[X^2]
$$

而：

$$
f(\mathbb E[X])=(\mathbb E[X])^2
$$

由于：

$$
\operatorname{Var}(X)
=
\mathbb E[X^2]-(\mathbb E[X])^2
$$

所以：

$$
\boxed{
\mathbb E[f(X)]-f(\mathbb E[X])
=
\operatorname{Var}(X)
}
\tag{A.8}
$$

对于 CT：

$$
X_{\mathrm{CT}}=x-hZ
$$

虽然：

$$
\mathbb E[X_{\mathrm{CT}}\mid X_t=x]
=
x-hv^*
$$

但：

$$
\mathbb E[f(X_{\mathrm{CT}})]
$$

通常不严格等于：

$$
f(x-hv^*)
$$

这就是 CT 有限时间间隔偏差的来源之一。

---

## A.5 Jacobian-Vector Product（JVP）

设：

$$
f:\mathbb R^d\rightarrow\mathbb R^m
$$

其 Jacobian：

$$
J_f(x)
=
\begin{bmatrix}
\frac{\partial f_1}{\partial x_1}&\cdots&
\frac{\partial f_1}{\partial x_d}\\
\vdots&\ddots&\vdots\\
\frac{\partial f_m}{\partial x_1}&\cdots&
\frac{\partial f_m}{\partial x_d}
\end{bmatrix}
$$

对于向量：

$$
v\in\mathbb R^d
$$

JVP 定义为：

$$
\boxed{
J_f(x)v
}
\tag{A.9}
$$

它表示函数沿方向 $v$ 的方向导数：

$$
\boxed{
J_f(x)v
=
\left.
\frac{d}{dh}f(x+hv)
\right|_{h=0}
}
\tag{A.10}
$$

Consistency Model 的输运方程：

$$
\partial_tf+J_xfv=0
$$

只需要：

$$
J_xfv
$$

而不需要整个 Jacobian。

因此，可以使用自动微分的 JVP 计算。

---

## A.6 ODE 的局部误差与全局误差

考虑：

$$
\frac{dx}{dt}=v(x,t)
$$

Euler：

$$
x_{n+1}=x_n+hv(x_n,t_n)
$$

精确解的 Taylor 展开：

$$
x(t+h)
=
x(t)+hx'(t)+\frac{h^2}{2}x''(t)+O(h^3)
$$

因为：

$$
x'(t)=v(x,t)
$$

所以：

$$
x(t+h)
=
x(t)+hv(x,t)+O(h^2)
$$

因此 Euler 的单步局部截断误差：

$$
\boxed{
e_{\mathrm{local}}=O(h^2)
}
\tag{A.11}
$$

经过约：

$$
N=O(1/h)
$$

步后：

$$
\boxed{
e_{\mathrm{global}}=O(h)
}
\tag{A.12}
$$

对于二阶 Heun：

$$
e_{\mathrm{local}}=O(h^3)
$$

$$
e_{\mathrm{global}}=O(h^2)
$$

这解释了 CD 的 Theorem 1 为什么会出现：

$$
O((\Delta t)^p)
$$

这样的误差界。

---

# 附录 B：连续时间 Consistency Training 与 Stop Gradient

这一部分对应原论文 Appendix B 的主要理论思想。

## B.1 从离散一致性到连续时间输运方程

离散一致性：

$$
f(X_t,t)
=
f(X_{t-h},t-h)
$$

如果：

$$
X_{t-h}=X_t-hv(X_t,t)+O(h^2)
$$

则：

$$
f(X_t-hv,t-h)
=
f(X_t,t)
-h(\partial_tf+J_xfv)
+O(h^2)
$$

一致性要求两者相等：

$$
0=-h(\partial_tf+J_xfv)+O(h^2)
$$

除以 $h$：

$$
0=-(\partial_tf+J_xfv)+O(h)
$$

令：

$$
h\rightarrow0
$$

得到：

$$
\boxed{
\partial_tf+J_xfv=0
}
\tag{B.1}
$$

这说明离散一致性约束在连续时间极限下对应输运 PDE。

---

## B.2 为什么 Stop Gradient 很重要？

考虑平方 L2 CT：

$$
\mathcal L_{\mathrm{CT}}
=
\mathbb E\|f_\theta(X_t,t)-f_{\theta^-}(X_t-hZ,t-h)\|^2
$$

如果 Target Network 固定，那么对 Online Network 的梯度只来自第一个分支。

对于固定的 $X_t=x$，最优 Online Network 输出为：

$$
f_\theta^*(x,t)
=
\mathbb E[
f_{\theta^-}(x-hZ,t-h)
\mid X_t=x
]
$$

Taylor 展开：

$$
f_\theta^*(x,t)
=
f_{\theta^-}(x,t)
-h\left[
\partial_tf_{\theta^-}
+
J_xf_{\theta^-}v^*
\right]
+O(h^2)
$$

如果 Online 和 Target 达到理想固定点：

$$
f_\theta=f_{\theta^-}=f
$$

则：

$$
0=
-h(\partial_tf+J_xfv^*)+O(h^2)
$$

连续时间极限下：

$$
\boxed{
\partial_tf+J_xfv^*=0
}
\tag{B.2}
$$

这解释了 CT 为什么能够在期望意义下学习正确的输运关系。

---

## B.3 如果两个分支都反向传播，会发生什么？

考虑：

$$
\mathcal L
=
\mathbb E\|f(X_t,t)-f(X_t-hZ,t-h)\|^2
$$

Taylor 展开：

$$
f(X_t,t)-f(X_t-hZ,t-h)
=
h(\partial_tf+J_xfZ)+O(h^2)
$$

因此：

$$
\frac{\mathcal L}{h^2}
\rightarrow
\mathbb E\|\partial_tf+J_xfZ\|^2
$$

定义：

$$
v^*=\mathbb E[Z\mid X_t=x]
$$

$$
\eta=Z-v^*
$$

则：

$$
\mathbb E[\eta\mid X_t=x]=0
$$

因此：

$$
\partial_tf+J_xfZ
=
(\partial_tf+J_xfv^*)+J_xf\eta
$$

平方：

$$
\begin{aligned}
\|\partial_tf+J_xfZ\|^2
={}&
\|\partial_tf+J_xfv^*\|^2\\
&+
\|J_xf\eta\|^2\\
&+
2(\partial_tf+J_xfv^*)^TJ_xf\eta
\end{aligned}
$$

固定 $X_t=x$ 取条件期望。

交叉项为零：

$$
\mathbb E[\eta\mid X_t=x]=0
$$

所以：

$$
\boxed{
\begin{aligned}
&\mathbb E[
\|\partial_tf+J_xfZ\|^2
\mid X_t=x
]\\
&=
\|\partial_tf+J_xfv^*\|^2\\
&\quad+
\operatorname{Tr}\left[
J_xf\operatorname{Cov}(Z\mid X_t=x)(J_xf)^T
\right]
\end{aligned}
}
\tag{B.3}
$$

第二项是非负的。

因此，如果两个分支都参与梯度计算，训练目标不只是要求：

$$
\partial_tf+J_xfv^*=0
$$

还会受到额外 Jacobian 项的影响。

这说明：

**Stop Gradient 不只是一个训练稳定性技巧，它还会改变优化目标的数学结构。**

原论文 Appendix B 的连续时间分析正是为了进一步研究这种区别。

---

# 附录 C：与 Flow Matching 的统一理解

考虑一般 Conditional Flow Matching：

$$
X_t=(1-t)X_0+tX_1
$$

条件速度：

$$
U=X_1-X_0
$$

边缘速度：

$$
\boxed{
v^*(x,t)=\mathbb E[U\mid X_t=x]
}
\tag{C.1}
$$

Flow Matching 直接训练：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E\|v_\theta(X_t,t)-U\|^2
}
\tag{C.2}
$$

其最优解：

$$
v_\theta^*(x,t)
=
\mathbb E[U\mid X_t=x]
$$

而 Consistency Model 学习：

$$
\boxed{
f_\theta(x,t)=\Phi_{t\rightarrow1}(x)
}
\tag{C.3}
$$

满足：

$$
\boxed{
\partial_tf+J_xfv^*=0
}
\tag{C.4}
$$

因此，可以将两者统一理解为：

**Flow Matching 学习概率流的局部速度。**

**Consistency Model 学习该概率流的全局终点映射。**

如果已有预训练 Flow Matching 模型：

$$
v_\phi(x,t,c)
$$

那么可以使用 CD 构造条件 Consistency Model：

$$
f_\theta(x,t,c)
$$

从而尝试实现一步轨迹生成。

这对实时自动驾驶规划具有直接研究价值，但属于对原论文的延伸，而不是原论文已经验证的应用。

---

## 最后总结

这篇论文最重要的数学关系只有三层。

**第一层：PF ODE 定义概率流。**

$$
\boxed{
\frac{dX_t}{dt}=v^*(X_t,t)
}
$$

**第二层：Consistency Function 学习该概率流的终点映射。**

$$
\boxed{
f^*(x,t)=\Phi_{t\rightarrow\epsilon}(x)
}
$$

**第三层：利用局部一致性学习全局映射。**

CD 使用教师 ODE：

$$
\boxed{
X_{t_{n+1}}
\xrightarrow{\mathrm{ODE}}
\widehat X_{t_n}
}
$$

CT 使用随机条件路径：

$$
\boxed{
X_t=X_0+tZ
}
$$

并依赖：

$$
\boxed{
\mathbb E[Z\mid X_t=x]=v^*(x,t)
}
$$

使其局部训练关系在期望和连续时间极限下与正确的边缘动力学联系起来。

**因此，Consistency Models 的本质不是设计一个更高阶的 ODE Solver，而是通过一致性约束，直接学习 ODE Solver 原本需要通过多步积分才能得到的全局流映射。**